# 03 · Engine v2 demo (v1 vs v2 side by side)

Runs `ml/recommender_v2.py` on O*NET 31.0 and compares it with the v1 baseline.

**The profiles below are hand-made test profiles, not real students.** The numbers are sanity checks,
not accuracy results (there is no student data yet).

What v2 changes: readiness instead of correlation for skills and subjects, a score **range** when an answer
is "Can't assess", ranking by the lower end of the range, and at most 2 careers per field in the top 5.

In [ ]:
import os
import sys
from pathlib import Path
import pandas as pd

sys.path.append(str(Path("..").resolve()))   # lets the notebook find the ml folder
from ml.baseline_recommender import recommend_v1
from ml.recommender_v2 import load_onet_v2, check_maps, recommend_v2, explain_v2, target_career

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 20)

# The real O*NET folder; PATHFINDER_ONET_DIR can point somewhere else (used only to test this notebook).
ONET_DIR = Path(os.environ.get("PATHFINDER_ONET_DIR", Path("..") / "data" / "raw" / "onet"))
data = load_onet_v2(ONET_DIR)

print("Occupations we can rank :", len(data["codes"]))
print("Missing mapped elements :", check_maps(data))       # should be empty lists
for line in data["extras_info"]:
    print("Extra file              :", line)

## The two hand-made test profiles

The same profiles as notebook 02 (v1 format: 16 skills, 10 subjects). For v2 they are rewritten in the
survey format (8 skill rows, 8 subject rows). Merged rows take the mean of the v1 items they replace:
Physics/Chemistry = mean(Physics, Chemistry), Commerce/Economics = mean(Economics, Commerce/Accounting).

In [ ]:
# Test profile A: biology and research (hand-made test profile, not a real student)
profile_a_v1 = {
    "interests": {"Realistic": 2.8, "Investigative": 4.6, "Artistic": 3.4,
                  "Social": 3.2, "Enterprising": 2.1, "Conventional": 3.0},
    "skills": {"Mathematics": 3, "Programming": 3, "Problem Solving": 4, "Logical Reasoning": 4,
               "Critical Thinking": 4, "Research": 5, "Communication": 3, "Writing": 4,
               "Public Speaking": 2, "Leadership": 2, "Teamwork": 3, "Time Management": 3,
               "Project Management": 2, "Financial/Business Skills": 1, "Teaching": 3,
               "Technical/Practical Skills": 3},
    "subjects": {"Mathematics": 3, "Physics": 3, "Chemistry": 4, "Biology": 5,
                 "Computer Science": 4, "Economics": 2, "Commerce/Accounting": 1,
                 "Languages": 3, "Social Sciences": 2, "Arts/Design": 3},
}
# Test profile B: arts and communication (hand-made test profile, not a real student)
profile_b_v1 = {
    "interests": {"Realistic": 1.8, "Investigative": 2.5, "Artistic": 4.4,
                  "Social": 3.9, "Enterprising": 4.2, "Conventional": 2.6},
    "skills": {"Mathematics": 2, "Programming": 1, "Problem Solving": 3, "Logical Reasoning": 3,
               "Critical Thinking": 3, "Research": 3, "Communication": 5, "Writing": 4,
               "Public Speaking": 4, "Leadership": 4, "Teamwork": 4, "Time Management": 3,
               "Project Management": 3, "Financial/Business Skills": 3, "Teaching": 3,
               "Technical/Practical Skills": 1},
    "subjects": {"Mathematics": 2, "Physics": 1, "Chemistry": 1, "Biology": 2,
                 "Computer Science": 2, "Economics": 3, "Commerce/Accounting": 3,
                 "Languages": 5, "Social Sciences": 4, "Arts/Design": 5},
}

SKILL_ROWS = ["Mathematics", "Programming", "Problem Solving", "Research",
              "Communication", "Writing", "Leadership", "Technical/Practical Skills"]

def to_v2(p):
    """Rewrite a v1 test profile in the v2 survey format (every answer rated)."""
    s = p["subjects"]
    subjects = {"Mathematics": s["Mathematics"], "Physics/Chemistry": (s["Physics"] + s["Chemistry"]) / 2,
                "Biology": s["Biology"], "Computer Science": s["Computer Science"],
                "Commerce/Economics": (s["Economics"] + s["Commerce/Accounting"]) / 2,
                "Languages": s["Languages"], "Social Sciences": s["Social Sciences"], "Arts/Design": s["Arts/Design"]}
    return {"interests": p["interests"],
            "skills": {row: {"rating": p["skills"][row]} for row in SKILL_ROWS},
            "subjects": {row: {"rating": v} for row, v in subjects.items()}}

profile_a, profile_b = to_v2(profile_a_v1), to_v2(profile_b_v1)
profile_a

## v1 and v2 side by side (top 5, Job Zone >= 3)

In [ ]:
def side_by_side(p_v1, p_v2):
    v1 = recommend_v1(p_v1, data, top_n=5, min_job_zone=3)
    v2 = recommend_v2(p_v2, data, top_n=5, min_job_zone=3)
    left = v1[["rank", "career", "domain", "match_score"]].set_axis(
        ["rank", "v1 career", "v1 field", "v1 score"], axis=1)
    right = v2[["rank", "career", "field", "match_low", "display"]].set_axis(
        ["rank", "v2 career", "v2 field", "v2 score", "v2 display"], axis=1)
    return left.merge(right, on="rank", how="outer").set_index("rank")   # joined by rank, not row label

print("Profile A (hand-made test profile: biology and research)")
side_by_side(profile_a_v1, profile_a)

In [ ]:
print("Profile B (hand-made test profile: arts and communication)")
side_by_side(profile_b_v1, profile_b)

## Why, and what to learn (v2)

Reasons come only from the profile's own answers. Gaps: importance >= 3 and more than 0.5 levels below
the required level, most urgent first.

In [ ]:
recs_a = recommend_v2(profile_a, data, top_n=5)
for _, row in recs_a.head(3).iterrows():
    reasons, gaps = explain_v2(profile_a, data, row["code"])
    print(f"\n#{row['rank']}  {row['career']}  -  {row['match_low']}/100  ({row['field']})")
    for r in reasons or ["(overall profile)"]:
        print("   +", r)
    print(gaps.head(4).to_string(index=False) if len(gaps) else "   no important gaps")

## Profile C: profile A with two "Can't assess" answers

Research (a skill row) and Biology (a subject row) are set to "Can't assess" (hand-made test profile).
The engine now shows a range: the low end assumes level 0, the high end assumes the requirement is met.
Careers are ranked by the low end, so not answering can never move a career up.

In [ ]:
profile_c = {**profile_a,
             "skills": {**profile_a["skills"], "Research": "cant_assess"},
             "subjects": {**profile_a["subjects"], "Biology": "cant_assess"}}
recs_c = recommend_v2(profile_c, data, top_n=5)
recs_c[["rank", "career", "field", "match_low", "match_high", "coverage", "display",
        "critical_unknowns", "follow_ups"]]

In [ ]:
# Compare with the full answers: the low end is never above the full score,
# and the high end is never below it.
full = recommend_v2(profile_a, data, top_n=len(data["codes"]), diversity=False).set_index("code")
check = recs_c.set_index("code")[["career", "match_low", "match_high"]].join(full["match_low"].rename("full answers"))
check

In [ ]:
# Aiming at the first career of profile C: which questions are still needed?
target = target_career(profile_c, data, recs_c.iloc[0]["code"])
print("Career      :", target["career"], "-", target["field"])
print("Readiness   :", target["readiness"])
print("Interest fit:", target["interest_fit"])
print("Questions   :", [(q["category"], q["item"]) for q in target["questions_needed"]])
print("Alternatives:")
pd.DataFrame(target["alternatives"])